# 《On the Closed-Form of Flow Matching》全文详解
## Generalization Does Not Arise from Target Stochasticity

**论文：** [On the Closed-Form of Flow Matching](https://arxiv.org/abs/2506.03719?utm_source=chatgpt.com)  
**作者：** Quentin Bertrand、Anne Gagneux、Mathurin Massias、Rémi Emonet  
**版本：** arXiv v2，2025 年 12 月 1 日  
**代码：** [官方 GitHub 仓库](https://github.com/generativemodels/closedformfm?utm_source=chatgpt.com)


下面按照**核心结论 → 数学基础 → 理论推导 → 论文实验 → EFM 方法 → 证明 → 局限与启发**的顺序整理。公式重新统一编号，并在首次出现时解释符号。

---

# 一、全文核心结论

这篇论文研究的不是如何提高 Flow Matching 的采样速度，而是一个更基础的问题：

> **为什么 Flow Matching 能够生成训练集之外的新样本？这种泛化能力是否来自训练 Target 的随机性？**

作者的答案是：

**在论文研究的高维图像生成任务中，条件速度 Target 的随机性并不是泛化能力的关键来源。泛化与有限容量神经网络无法精确拟合训练集对应的经验最优速度场密切相关，而这种差异在生成早期尤其重要。**

全文可以归纳为六个核心结论。

### 结论 1：有限训练集对应的最优 FM 速度场具有闭式解

给定训练集：

$$
\mathcal D=\{y_1,\ldots,y_N\}
$$

以及标准线性插值：

$$
X_t=(1-t)X_0+tX_1
$$

可以直接计算经验分布下的最优边缘速度：

$$
\boxed{
\hat v^*(x,t)
=
\frac{
\sum_{i=1}^Nw_i(x,t)y_i-x
}{1-t}
}
\tag{1.1}
$$

其中：

$$
\boxed{
w_i(x,t)
=
\operatorname{Softmax}_i
\left(
-\frac{\|x-ty_i\|^2}{2(1-t)^2}
\right)
}
\tag{1.2}
$$

这个闭式解是全文的理论基础。

但它对应的是**有限训练集的经验分布**，不是真实数据分布。

### 结论 2：精确经验速度场会记忆训练集

如果使用式 (1.1) 对应的精确概率流，从标准高斯分布生成数据，那么在理想连续时间极限下：

$$
\boxed{
p_1
=
\frac1N\sum_{i=1}^N\delta_{y_i}
}
\tag{1.3}
$$

最终生成分布只包含训练样本。

因此：

**精确拟合经验最优速度场，并不等于获得好的生成泛化能力。**

这也是论文提出的核心矛盾：

$$
\boxed{
\begin{gathered}
\text{精确经验速度场会记忆训练集}\\
\text{但实际训练的神经网络能够生成新样本}
\end{gathered}
}
$$

### 结论 3：高维数据中，随机条件速度很早就接近确定性边缘速度

标准 CFM 使用随机 Target：

$$
V=X_1-X_0
$$

但真正希望学习的是：

$$
\hat v^*(x,t)
=
\mathbb E[V\mid X_t=x]
$$

作者发现，在 CIFAR-10 等高维图像数据上，两者的方向在生成早期就已经高度一致。

换句话说，虽然 CFM 的 Target 在数学上是随机的，但在高维数据的很多时间区间内，它实际上已经非常接近一个确定性 Target。

这是 Figure 1 的主要结论。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

### 结论 4：泛化与神经网络无法拟合精确经验速度场有关

作者固定网络架构，只改变训练集大小。

发现：

- 训练集很小时，网络容易拟合精确经验速度场，生成结果也容易记忆训练集。
- 训练集变大后，精确经验速度场更加复杂，网络无法完全拟合。
- 在这种情况下，模型开始表现出更好的生成泛化能力。

需要强调：

**论文证明的是实验相关性，而不是一个普遍定理：只要拟合失败就一定能够泛化。**

这是 Figure 2 的核心结论。

### 结论 5：泛化相关的速度场差异主要发生在生成早期

作者构造混合速度场：

$$
v_{\mathrm{hybrid}}(x,t)=
\begin{cases}
\hat v^*(x,t),&t<\tau\\
v_\theta(x,t),&t\ge\tau
\end{cases}
$$

即前半段使用精确经验速度场，后半段使用神经网络速度场。

结果发现：

- $\tau\le0.2$：模型仍然能够生成较新的样本；
- $\tau>0.2$：生成结果开始明显接近训练集；
- $\tau\ge0.4$：结果往往已经非常接近精确经验流生成的训练样本。

因此，作者认为**生成早期的速度场近似对于泛化尤其重要**。

这是 Figure 3 的主要结论。[arxiv.org](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

### 结论 6：降低 Target 随机性，并不会损害泛化

作者提出 Empirical Flow Matching，简称 EFM。

它利用多个训练样本构造更接近闭式边缘速度的监督目标：

$$
\boxed{
\hat v_M(x,t)
=
\sum_{j=1}^M
\tilde w_j(x,t)
\frac{b_j-x}{1-t}
}
\tag{1.4}
$$

其中 $M$ 是参与构造 Target 的候选样本数量。

作者证明：

$$
\boxed{
\mathbb E[\hat v_M\mid X_t=x]
=
\hat v^*(x,t)
}
\tag{1.5}
$$

而且：

$$
\boxed{
\operatorname{Var}(\hat v_M\mid X_t=x)
\le
\operatorname{Var}(V\mid X_t=x)
}
\tag{1.6}
$$

实验表明，降低 Target 随机性没有导致生成质量下降，反而通常获得了小幅改善。

这是 Proposition 2 和 Figure 4 的核心结论。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

## 全文逻辑概览

| 论文部分 | 核心问题 | 主要结果 |
|---|---|---|
| Section 2 | 最优 FM 速度场能否解析计算？ | Proposition 1：闭式速度场 |
| Section 3.1 | CFM Target 的随机性有多大？ | 高维数据中很早出现近似确定性 |
| Section 3.2 | 为什么网络没有记忆所有训练样本？ | 网络对经验速度场的不完全拟合与泛化相关 |
| Section 3.3 | 泛化主要发生在哪个时间段？ | 生成早期尤其关键 |
| Section 4.1 | 能否降低 Target 随机性？ | EFM 和 Proposition 2 |
| Section 4.2 | 降低随机性是否损害泛化？ | 没有，部分指标反而改善 |

接下来逐步推导。

---

# 二、数学基础：CFM 究竟在学习什么？

## 2.1 定义数据分布与随机变量

首先定义：

$$
X_0\sim p_0=\mathcal N(0,I)
$$

其中：

- $X_0\in\mathbb R^D$：初始高斯噪声；
- $D$：数据维度；
- $I$：$D\times D$ 单位矩阵。

真实数据分布为：

$$
X_1\sim p_{\mathrm{data}}
$$

其中 $X_1\in\mathbb R^D$ 是真实数据样本。

假设：

$$
X_0\perp X_1
$$

即二者独立。

Flow Matching 的目标是学习速度场：

$$
v_\theta(x,t):\mathbb R^D\times[0,1]\rightarrow\mathbb R^D
$$

使得求解 ODE：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(X_t,t)
}
\tag{2.1}
$$

能够将初始分布：

$$
p_0
$$

输运到数据分布：

$$
p_{\mathrm{data}}
$$

这里要注意：

**ODE 中的 $X_t$ 是按照学习到的速度场演化的状态，而训练时构造的 $X_t$ 是通过插值采样得到的状态。**

当速度场精确时，两者在每个时间的边缘分布一致，但单条 ODE 轨迹不一定等于训练时随机配对的直线。

---

## 2.2 定义条件概率路径

采用 OT-CFM 常见的线性插值路径：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{2.2}
$$

其中：

$$
t\in[0,1]
$$

满足：

$$
X_{t=0}=X_0
$$

$$
X_{t=1}=X_1
$$

对时间求导：

$$
\frac{dX_t}{dt}
=
-X_0+X_1
$$

所以：

$$
\boxed{
V=X_1-X_0
}
\tag{2.3}
$$

这里 $V$ 是训练样本对应的条件速度。

由于每次训练都会随机采样：

$$
(X_0,X_1)
$$

所以 $V$ 是随机变量。

---

## 2.3 为什么条件速度不等于边缘速度？

假设当前状态：

$$
X_t=x
$$

由：

$$
x=(1-t)X_0+tX_1
$$

可以解出：

$$
X_0=\frac{x-tX_1}{1-t}
$$

代入条件速度：

$$
\begin{aligned}
V
&=X_1-X_0\\
&=X_1-\frac{x-tX_1}{1-t}\\
&=\frac{X_1-x}{1-t}
\end{aligned}
$$

因此：

$$
\boxed{
V=\frac{X_1-x}{1-t}
\quad\text{当 }X_t=x
}
\tag{2.4}
$$

注意：给定 $X_t=x$ 后，$X_1$ 仍然可能是不确定的。

因此，条件速度 $V$ 仍然具有随机性。

真正的边缘速度定义为：

$$
\boxed{
v^*(x,t)
=
\mathbb E[V\mid X_t=x]
}
\tag{2.5}
$$

代入式 (2.4)：

$$
\begin{aligned}
v^*(x,t)
&=
\mathbb E
\left[
\frac{X_1-x}{1-t}
\middle|X_t=x
\right]\\
&=
\frac{
\mathbb E[X_1\mid X_t=x]-x
}{1-t}
\end{aligned}
$$

因此：

$$
\boxed{
v^*(x,t)
=
\frac{
\mathbb E[X_1\mid X_t=x]-x
}{1-t}
}
\tag{2.6}
$$

这就是全文最重要的基础公式。

它说明：

**只要能够计算给定当前状态 $x$ 后，最终数据 $X_1$ 的后验均值，就能够得到边缘速度。**

---

# 三、为什么边缘速度能够驱动正确的概率流？

论文直接引用了 CFM 的已有理论。为了理解后续结论，这里补充推导。

## 3.1 条件路径的概率密度

给定：

$$
X_1=y
$$

那么：

$$
X_t=(1-t)X_0+ty
$$

由于：

$$
X_0\sim\mathcal N(0,I)
$$

得到：

$$
\boxed{
X_t\mid X_1=y
\sim
\mathcal N(ty,(1-t)^2I)
}
\tag{3.1}
$$

记条件密度为：

$$
p_t(x\mid y)
$$

对应的条件速度：

$$
\boxed{
v^{\mathrm{cond}}(x,t\mid y)
=
\frac{y-x}{1-t}
}
\tag{3.2}
$$

---

## 3.2 条件连续性方程

对于确定性 ODE：

$$
\dot x=v(x,t)
$$

其概率密度演化满足连续性方程：

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
+
\nabla_x\cdot
\left(
p_t(x)v(x,t)
\right)
=0
}
\tag{3.3}
$$

这里：

- $\nabla_x$：对状态 $x$ 求梯度；
- $\nabla_x\cdot$：散度；
- $p_tv$：概率流密度。

直观上，它表示概率质量不会凭空产生或消失，只能随着速度场移动。

对于每一个固定的终点 $y$，条件概率路径满足：

$$
\boxed{
\partial_t p_t(x\mid y)
+
\nabla_x\cdot
\left[
p_t(x\mid y)
v^{\mathrm{cond}}(x,t\mid y)
\right]
=0
}
\tag{3.4}
$$

---

## 3.3 对终点变量边缘化

整体边缘密度为：

$$
\boxed{
p_t(x)
=
\int
p_t(x\mid y)p_{\mathrm{data}}(y)\,dy
}
\tag{3.5}
$$

对时间求导：

$$
\partial_t p_t(x)
=
\int
\partial_t p_t(x\mid y)
p_{\mathrm{data}}(y)\,dy
$$

将式 (3.4) 代入：

$$
\begin{aligned}
\partial_t p_t(x)
&=
-\int
\nabla_x\cdot
\left[
p_t(x\mid y)
v^{\mathrm{cond}}(x,t\mid y)
\right]
p_{\mathrm{data}}(y)\,dy
\end{aligned}
$$

在适当正则性条件下交换散度与积分：

$$
\partial_t p_t(x)
=
-\nabla_x\cdot
\left[
\int
p_t(x\mid y)
v^{\mathrm{cond}}(x,t\mid y)
p_{\mathrm{data}}(y)\,dy
\right]
$$

根据贝叶斯公式：

$$
p(y\mid x,t)
=
\frac{
p_t(x\mid y)p_{\mathrm{data}}(y)
}{
p_t(x)
}
$$

因此：

$$
p_t(x\mid y)p_{\mathrm{data}}(y)
=
p_t(x)p(y\mid x,t)
$$

代入：

$$
\begin{aligned}
\partial_t p_t(x)
&=
-\nabla_x\cdot
\left[
p_t(x)
\int
v^{\mathrm{cond}}(x,t\mid y)
p(y\mid x,t)\,dy
\right]
\end{aligned}
$$

定义：

$$
\boxed{
v^*(x,t)
=
\int
v^{\mathrm{cond}}(x,t\mid y)
p(y\mid x,t)\,dy
}
\tag{3.6}
$$

于是：

$$
\boxed{
\partial_t p_t(x)
+
\nabla_x\cdot
\left[
p_t(x)v^*(x,t)
\right]
=0
}
\tag{3.7}
$$

这说明：

**条件速度场经过后验加权平均，得到的边缘速度场能够驱动我们预先定义的边缘概率路径。**

注意，这里保证的是边缘分布演化，而不是要求每一条 ODE 轨迹都是：

$$
X_t=(1-t)X_0+tX_1
$$

这一区别在研究 Reflow、MeanFlow 和 Shortcut 时同样重要。

---

# 四、Proposition 1：经验最优速度场的闭式解

这一部分是论文 Section 2 的核心。

## 4.1 将真实数据分布替换为经验分布

真实数据分布：

$$
p_{\mathrm{data}}
$$

通常未知。

我们实际拥有的是训练集：

$$
\mathcal D=\{y_1,\ldots,y_N\}
$$

其中：

$$
y_i\in\mathbb R^D
$$

对应经验分布：

$$
\boxed{
\hat p_{\mathrm{data}}(y)
=
\frac1N\sum_{i=1}^N\delta(y-y_i)
}
\tag{4.1}
$$

这里 $\delta$ 是 Dirac 分布。

因此，随机终点 $X_1$ 满足：

$$
\boxed{
P(X_1=y_i)=\frac1N
}
\tag{4.2}
$$

---

## 4.2 写出条件高斯密度

根据式 (3.1)：

$$
X_t\mid X_1=y_i
\sim
\mathcal N(ty_i,(1-t)^2I)
$$

因此：

$$
\boxed{
p_t(x\mid y_i)
=
\frac1{[2\pi(1-t)^2]^{D/2}}
\exp\left[
-\frac{\|x-ty_i\|^2}{2(1-t)^2}
\right]
}
\tag{4.3}
$$

这一步是把已知的高斯分布真正写成概率密度函数。

---

## 4.3 使用贝叶斯公式计算后验

我们需要计算：

$$
P(X_1=y_i\mid X_t=x)
$$

根据贝叶斯公式：

$$
\boxed{
P(X_1=y_i\mid X_t=x)
=
\frac{
p_t(x\mid y_i)P(X_1=y_i)
}{
p_t(x)
}
}
\tag{4.4}
$$

这里分子中的：

$$
P(X_1=y_i)=\frac1N
$$

而分母通过全概率公式得到：

$$
\boxed{
p_t(x)
=
\frac1N\sum_{j=1}^Np_t(x\mid y_j)
}
\tag{4.5}
$$

所以：

$$
\begin{aligned}
P(X_1=y_i\mid X_t=x)
&=
\frac{
\frac1N p_t(x\mid y_i)
}{
\frac1N\sum_jp_t(x\mid y_j)
}\\
&=
\frac{
p_t(x\mid y_i)
}{
\sum_jp_t(x\mid y_j)
}
\end{aligned}
$$

将式 (4.3) 代入。

因为所有条件高斯的协方差都相同，所以归一化系数：

$$
\frac1{[2\pi(1-t)^2]^{D/2}}
$$

在分子分母中抵消。

最终得到：

$$
\boxed{
w_i(x,t)
=
\frac{
\exp\left[
-\frac{\|x-ty_i\|^2}{2(1-t)^2}
\right]
}{
\displaystyle
\sum_{j=1}^N
\exp\left[
-\frac{\|x-ty_j\|^2}{2(1-t)^2}
\right]
}
}
\tag{4.6}
$$

其中：

$$
\boxed{
w_i(x,t)
=
P(X_1=y_i\mid X_t=x)
}
$$

因此：

$$
\sum_{i=1}^Nw_i(x,t)=1
$$

---

## 4.4 代入边缘速度公式

根据式 (2.6)：

$$
v^*(x,t)
=
\frac{
\mathbb E[X_1\mid X_t=x]-x
}{1-t}
$$

而：

$$
\mathbb E[X_1\mid X_t=x]
=
\sum_{i=1}^Nw_i(x,t)y_i
$$

所以：

$$
\begin{aligned}
\hat v^*(x,t)
&=
\frac{
\sum_iw_i(x,t)y_i-x
}{1-t}\\
&=
\frac{
\sum_iw_i(x,t)y_i
-
\sum_iw_i(x,t)x
}{1-t}\\
&=
\sum_iw_i(x,t)
\frac{y_i-x}{1-t}
\end{aligned}
$$

最终：

$$
\boxed{
\hat v^*(x,t)
=
\sum_{i=1}^N
w_i(x,t)
\frac{y_i-x}{1-t}
}
\tag{4.7}
$$

这就是论文的 Proposition 1。

### 这里的“最优”究竟是什么意思？

必须区分两个对象：

真实分布对应的最优速度：

$$
v_{\mathrm{population}}^*(x,t)
$$

与训练集经验分布对应的最优速度：

$$
\hat v^*(x,t)
$$

论文能够计算的是后者。

因此：

$$
\boxed{
\hat v^*(x,t)
\neq
v_{\mathrm{population}}^*(x,t)
}
$$

一般情况下二者并不相同。

而且，这里的“最优”指的是**给定 FM 概率路径下，平方损失的最优回归目标**，不是指最优传输代价意义上的全局最优速度场。

---

# 五、为什么精确经验速度场会记忆训练集？

这是论文第一个重要理论推论。

## 5.1 经验概率路径是高斯混合分布

根据：

$$
p_t(x)
=
\frac1N\sum_i p_t(x\mid y_i)
$$

得到：

$$
\boxed{
p_t(x)
=
\frac1N
\sum_{i=1}^N
\mathcal N
\left(
x;ty_i,(1-t)^2I
\right)
}
\tag{5.1}
$$

这里用 $\mathcal N(x;\mu,\Sigma)$ 表示在 $x$ 处的高斯密度。

当：

$$
t=0
$$

时：

$$
p_0(x)=\mathcal N(x;0,I)
$$

当：

$$
t\rightarrow1
$$

时：

$$
ty_i\rightarrow y_i
$$

同时：

$$
(1-t)^2I\rightarrow0
$$

所以：

$$
\boxed{
p_t
\xrightarrow[t\rightarrow1]{}
\frac1N\sum_{i=1}^N\delta_{y_i}
}
\tag{5.2}
$$

这里的收敛是分布意义下的弱收敛。

因此，精确经验速度场对应的概率流，最终会将高斯分布输运到训练集经验分布。

---

## 5.2 从速度场本身理解记忆现象

闭式速度为：

$$
\hat v^*(x,t)
=
\sum_iw_i(x,t)\frac{y_i-x}{1-t}
$$

当：

$$
t\rightarrow1
$$

时，Softmax 中的分母：

$$
2(1-t)^2
$$

趋近于零。

对于固定的、具有唯一最近邻的状态 $x$，权重将趋向于距离最近的训练样本。

记：

$$
i^*=\arg\min_i\|x-y_i\|^2
$$

则通常有：

$$
w_{i^*}(x,t)\rightarrow1
$$

其他权重趋向于零。

于是：

$$
\boxed{
\hat v^*(x,t)
\approx
\frac{y_{i^*}-x}{1-t}
}
\tag{5.3}
$$

这表示速度场越来越强烈地指向某个训练样本。

因此，精确经验速度场在终点附近具有明显的样本吸引结构。

不过需要注意：这是固定 $x$ 下的渐近分析，不能直接把它当成每条真实 ODE 轨迹的逐点近似证明。

真正保证终端分布为经验分布的是式 (5.1)～(5.2) 的概率流性质。

---

## 5.3 由此产生的核心矛盾

如果神经网络能够精确学习：

$$
v_\theta(x,t)=\hat v^*(x,t)
$$

那么：

$$
p_\theta(x_1)=\hat p_{\mathrm{data}}(x_1)
$$

即最终生成训练样本。

但实际训练的 Flow Matching 模型通常可以生成训练集之外的新图像。

因此：

$$
\boxed{
\begin{gathered}
\text{为什么模型没有学到精确经验速度场？}\\
\text{这种偏差又是如何影响泛化的？}
\end{gathered}
}
$$

这是论文 Section 3 的出发点。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 六、CFM Loss 与闭式 FM Loss 的数学关系

在进入实验之前，我们先推导一个关键的统计关系。

这一部分是对论文理论基础的展开，有助于理解作者为什么要研究 Target Stochasticity。

## 6.1 定义两个训练目标

标准 CFM：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}(\theta)
=
\mathbb E
\left[
\|v_\theta(X_t,t)-V\|^2
\right]
}
\tag{6.1}
$$

其中：

$$
V=X_1-X_0
$$

闭式 FM：

$$
\boxed{
\mathcal L_{\mathrm{FM}}(\theta)
=
\mathbb E
\left[
\|v_\theta(X_t,t)-\hat v^*(X_t,t)\|^2
\right]
}
\tag{6.2}
$$

其中：

$$
\hat v^*(X_t,t)
=
\mathbb E[V\mid X_t,t]
$$

两个 Loss 的区别是：

- CFM 的 Target 是随机变量 $V$；
- FM 的 Target 是确定性函数 $\hat v^*(X_t,t)$。

---

## 6.2 将随机 Target 分解为均值与残差

定义：

$$
\boxed{
\epsilon
=
V-\hat v^*(X_t,t)
}
\tag{6.3}
$$

那么：

$$
V=\hat v^*(X_t,t)+\epsilon
$$

根据条件期望的性质：

$$
\boxed{
\mathbb E[\epsilon\mid X_t,t]=0
}
\tag{6.4}
$$

将其代入 CFM：

$$
\begin{aligned}
\mathcal L_{\mathrm{CFM}}
&=
\mathbb E
\left[
\|v_\theta-\hat v^*-\epsilon\|^2
\right]\\
&=
\mathbb E\left[
\|v_\theta-\hat v^*\|^2
\right]\\
&\quad+
\mathbb E[\|\epsilon\|^2]\\
&\quad-
2\mathbb E[
(v_\theta-\hat v^*)^\top\epsilon
]
\end{aligned}
\tag{6.5}
$$

交叉项：

$$
\begin{aligned}
&\mathbb E[
(v_\theta-\hat v^*)^\top\epsilon
]\\
&=
\mathbb E
\left[
(v_\theta-\hat v^*)^\top
\mathbb E[\epsilon\mid X_t,t]
\right]\\
&=0
\end{aligned}
$$

因此：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}(\theta)
=
\mathcal L_{\mathrm{FM}}(\theta)
+
C
}
\tag{6.6}
$$

其中：

$$
\boxed{
C=
\mathbb E
\left[
\|V-\hat v^*(X_t,t)\|^2
\right]
}
\tag{6.7}
$$

是与模型参数无关的常数。

所以：

$$
\boxed{
\arg\min_\theta\mathcal L_{\mathrm{CFM}}
=
\arg\min_\theta\mathcal L_{\mathrm{FM}}
}
\tag{6.8}
$$

这说明两种训练方式的期望目标相同。

但是，**相同的期望优化目标，并不意味着 SGD 训练轨迹相同**。

因为有限 batch 的梯度方差可能不同。

---

## 6.3 Target 随机性可以如何量化？

我们可以直接计算：

$$
\operatorname{Var}(V\mid X_t=x)
$$

这里使用向量总方差：

$$
\operatorname{Var}(V\mid X_t=x)
=
\mathbb E
\left[
\|V-\mathbb E[V\mid X_t=x]\|^2
\mid X_t=x
\right]
$$

根据：

$$
V=\frac{X_1-x}{1-t}
$$

得到：

$$
\boxed{
\operatorname{Var}(V\mid X_t=x)
=
\frac{
\operatorname{Tr}
\left[
\operatorname{Cov}(X_1\mid X_t=x)
\right]
}{(1-t)^2}
}
\tag{6.9}
$$

由于：

$$
P(X_1=y_i\mid X_t=x)=w_i(x,t)
$$

定义后验均值：

$$
\bar y(x,t)=\sum_iw_i(x,t)y_i
$$

那么：

$$
\boxed{
\operatorname{Var}(V\mid X_t=x)
=
\frac1{(1-t)^2}
\sum_i
w_i(x,t)
\|y_i-\bar y(x,t)\|^2
}
\tag{6.10}
$$

这是一个非常直观的结果。

如果后验权重：

$$
w_i(x,t)
$$

分散在多个样本上，那么条件速度的随机性可能较大。

如果：

$$
w_k(x,t)\approx1
$$

那么：

$$
\bar y(x,t)\approx y_k
$$

并且：

$$
\operatorname{Var}(V\mid X_t=x)\approx0
$$

因此：

**Target 随机性的大小，实际上取决于给定中间状态后，最终样本的后验不确定性。**

---

# 七、Section 3.1：为什么高维数据中的 Target 很快变得近似确定？

这是论文 Figure 1。

## 7.1 作者首先测量什么？

作者固定一组：

$$
(X_0,X_1)
$$

然后构造：

$$
X_t=(1-t)X_0+tX_1
$$

分别计算：

$$
V=X_1-X_0
$$

以及：

$$
\hat v^*(X_t,t)
$$

接着计算余弦相似度：

$$
\boxed{
C(X_t,t)
=
\frac{
V^\top\hat v^*(X_t,t)
}{
\|V\|\|\hat v^*(X_t,t)\|
}
}
\tag{7.1}
$$

其中：

$$
C\in[-1,1]
$$

如果：

$$
C\approx1
$$

说明两个速度方向高度一致。

作者对 256 个随机样本对进行统计，并在不同时间 $t$ 下绘制余弦相似度的分布。

实验使用：

- Two-Moons：二维数据；
- CIFAR-10：$D=3072$；
- Imagenette：通过改变图像分辨率研究维度影响。

Figure 1(a) 比较二维与高维数据，Figure 1(c) 则直接研究数据维度变化的影响。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

## 7.2 为什么数据维度会影响后验集中？

这一部分可以从闭式 Softmax 推导出来。

假设某个中间状态由第 $i$ 个训练样本生成：

$$
X_t=(1-t)X_0+ty_i
$$

我们希望比较：

$$
w_i(X_t,t)
$$

与另一个样本 $y_j$ 的权重：

$$
w_j(X_t,t)
$$

根据 Softmax：

$$
\frac{w_i}{w_j}
=
\exp
\left[
-\frac{
\|X_t-ty_i\|^2-\|X_t-ty_j\|^2
}{
2(1-t)^2
}
\right]
$$

定义两个样本的差：

$$
\Delta_{ij}=y_i-y_j
$$

那么：

$$
X_t-ty_i=(1-t)X_0
$$

以及：

$$
X_t-ty_j=(1-t)X_0+t\Delta_{ij}
$$

展开平方：

$$
\begin{aligned}
\|X_t-ty_j\|^2
&=
\|(1-t)X_0+t\Delta_{ij}\|^2\\
&=
(1-t)^2\|X_0\|^2\\
&\quad+
2t(1-t)X_0^\top\Delta_{ij}\\
&\quad+
t^2\|\Delta_{ij}\|^2
\end{aligned}
$$

而：

$$
\|X_t-ty_i\|^2
=
(1-t)^2\|X_0\|^2
$$

相减：

$$
\begin{aligned}
&\|X_t-ty_j\|^2-\|X_t-ty_i\|^2\\
&=
2t(1-t)X_0^\top\Delta_{ij}
+
t^2\|\Delta_{ij}\|^2
\end{aligned}
$$

所以：

$$
\boxed{
\log\frac{w_i}{w_j}
=
\frac{t}{1-t}X_0^\top\Delta_{ij}
+
\frac{t^2}{2(1-t)^2}\|\Delta_{ij}\|^2
}
\tag{7.2}
$$

这里有两项。

第一项：

$$
\frac{t}{1-t}X_0^\top\Delta_{ij}
$$

是随机项。

第二项：

$$
\frac{t^2}{2(1-t)^2}\|\Delta_{ij}\|^2
$$

是有利于真实样本 $y_i$ 的确定性项。

因为：

$$
X_0\sim\mathcal N(0,I)
$$

所以：

$$
X_0^\top\Delta_{ij}
\sim
\mathcal N(0,\|\Delta_{ij}\|^2)
$$

因此：

$$
\boxed{
\log\frac{w_i}{w_j}
\sim
\mathcal N
\left(
\frac{t^2\|\Delta_{ij}\|^2}{2(1-t)^2},
\frac{t^2\|\Delta_{ij}\|^2}{(1-t)^2}
\right)
}
\tag{7.3}
$$

这是我们根据论文闭式公式推导出的补充分析，并非论文单独陈述的定理。

---

## 7.3 高维为什么会加速后验集中？

假设样本之间的典型距离满足：

$$
\|\Delta_{ij}\|^2\propto D
$$

那么式 (7.3) 的均值：

$$
\mu\propto D
$$

标准差：

$$
\sigma\propto\sqrt D
$$

因此：

$$
\frac{\mu}{\sigma}
\propto\sqrt D
$$

随着维度增加，真实样本的后验对数权重优势可能越来越显著。

因此，在固定 $t>0$ 且样本间距随维度增长的条件下：

$$
w_i(X_t,t)\approx1
$$

其他候选样本：

$$
w_j(X_t,t)\approx0
$$

这时：

$$
\begin{aligned}
\hat v^*(X_t,t)
&=
\sum_jw_j(X_t,t)
\frac{y_j-X_t}{1-t}\\
&\approx
\frac{y_i-X_t}{1-t}\\
&=y_i-X_0\\
&=V
\end{aligned}
$$

所以：

$$
\boxed{
\hat v^*(X_t,t)\approx V
}
\tag{7.4}
$$

**这解释了高维数据中随机 Target 为什么可能迅速接近确定性 Target。**

但这个分析并不是对任意高维数据都成立的普遍结论。

它还取决于：

- 样本之间的距离；
- 数据是否集中在低维流形；
- 训练集大小 $N$；
- 当前时间 $t$。

特别是，如果训练集大小随维度指数增长，那么即使单个错误样本的权重很小，也可能有大量候选样本共同贡献后验概率。

---

## 7.4 Figure 1 的具体结论

作者发现：

**二维 Two-Moons：**

在较长一段时间内，随机条件速度与边缘速度仍然存在明显方向差异。

**CIFAR-10：**

在生成早期，两者就已经高度对齐。

论文指出，对于 CIFAR-10，近似非随机区域在 $t<0.2$ 时便已经出现。

**Imagenette：**

随着图像分辨率提高，条件速度与边缘速度达到高余弦相似度的时间明显提前。

Figure 1(c) 统计的是：

$$
\boxed{
P(C(X_t,t)>0.9)
}
\tag{7.5}
$$

而不是直接统计 Target 方差。

因此，严格来说，Figure 1 直接证明的是：

**高维数据中条件速度与边缘速度迅速发生方向对齐。**

结合闭式公式中的后验权重集中现象，作者将其解释为 Target 随机性迅速减弱。

---

# 八、Section 3.2：为什么神经网络没有学到精确经验速度场？

这一节是 Figure 2。

## 8.1 实验设计

作者使用 CIFAR-10。

固定一个约 3400 万参数的 U-Net。

改变训练集样本数量：

$$
\boxed{
N\in[10,10^4]
}
\tag{8.1}
$$

对于每个训练集大小，重新训练一个 CFM 模型：

$$
v_{\theta,N}(x,t)
$$

同时，根据当前训练集计算对应的精确经验速度场：

$$
\hat v_N^*(x,t)
$$

注意：

$$
\hat v_N^*
$$

随着训练集大小 $N$ 改变。

这是因为：

$$
\hat v_N^*(x,t)
=
\sum_{i=1}^N
w_i(x,t)
\frac{y_i-x}{1-t}
$$

包含了全部 $N$ 个训练样本。

---

## 8.2 指标一：速度场拟合误差

作者计算：

$$
\boxed{
E_N(t)
=
\mathbb E
\left[
\|v_{\theta,N}(X_t,t)-\hat v_N^*(X_t,t)\|^2
\right]
}
\tag{8.2}
$$

其中：

$$
X_t=(1-t)X_0+tX_1
$$

并且：

$$
X_1\sim\hat p_N
$$

所以，指标测量的是：

**在经验概率路径上的状态分布下，网络与精确经验速度场之间的均方误差。**

它并不测量整个 $\mathbb R^D$ 空间中所有状态的误差。

---

## 8.3 指标二：测试集 FID

作者计算生成分布与测试集之间的 FID。

这里使用 DINOv2 特征。

FID 主要反映生成样本与真实样本在特征空间中的统计相似程度。

越低越好。

但它不能单独判断模型是否复制训练样本。

---

## 8.4 指标三：生成样本与训练集的最近邻距离

对于生成样本 $x_{\mathrm{gen}}$，定义：

$$
\boxed{
d_{\mathrm{NN}}(x_{\mathrm{gen}})
=
\min_{y_i\in\mathcal D}
d(x_{\mathrm{gen}},y_i)
}
\tag{8.3}
$$

其中 $d$ 是实验选用的图像距离度量。

再对生成样本求平均：

$$
\boxed{
D_{\mathrm{NN}}
=
\mathbb E_{x_{\mathrm{gen}}}
[d_{\mathrm{NN}}(x_{\mathrm{gen}})]
}
\tag{8.4}
$$

如果：

$$
D_{\mathrm{NN}}\approx0
$$

说明生成样本非常接近训练集中的某些图像。

但距离较大也不必然意味着语义上的真正创新，因此该指标需要结合可视化和 FID 理解。

---

## 8.5 Figure 2 的实验结果

### 现象一：训练集很小时，网络容易拟合经验速度场

例如：

$$
N=10
$$

时，网络可以较好地拟合：

$$
\hat v_N^*
$$

但生成样本倾向于复制训练集。

这是因为经验速度场本身比较简单，只需要描述少数几个训练样本对应的输运结构。

### 现象二：训练集变大后，拟合误差增加

随着：

$$
N\uparrow
$$

经验速度场包含更多训练样本的吸引结构。

固定容量的神经网络难以完全拟合：

$$
\hat v_N^*
$$

因此：

$$
E_N(t)
$$

在部分时间区间明显增大。

### 现象三：拟合误差主要集中在两个时间段

作者观察到两个明显区域：

$$
\boxed{
t\approx0.15
}
$$

以及：

$$
\boxed{
t\rightarrow1
}
$$

后期误差容易理解。

因为：

$$
\hat v^*(x,t)
=
\frac{\bar y(x,t)-x}{1-t}
$$

当：

$$
t\rightarrow1
$$

时，分母趋近于零，经验速度场在训练样本附近呈现越来越尖锐的结构。

神经网络难以精确拟合。

但早期：

$$
t\approx0.15
$$

也出现显著误差，就非常有意思。

它恰好与 Figure 1 中后验开始快速集中的时间区域相近。

### 现象四：测试 FID 改善不一定意味着没有记忆

作者发现，使用约 1000 个训练样本时，模型的测试 FID 已经明显改善。

但最近邻距离仍然显示较强的训练样本记忆现象。

因此：

$$
\boxed{
\text{低测试 FID}
\not\Rightarrow
\text{没有记忆训练集}
}
\tag{8.5}
$$

---

## 8.6 这一节能够得出什么结论？

作者认为：

$$
\boxed{
\text{泛化与网络无法精确拟合经验速度场有关}
}
$$

但需要区分因果与相关性。

Figure 2 直接展示的是：

- 训练集大小变化；
- 精确经验速度场复杂度变化；
- 网络拟合误差变化；
- 生成质量和记忆程度变化。

它支持作者的解释，但没有单独证明：

$$
\text{拟合误差增大}
\Longrightarrow
\text{泛化能力提高}
$$

对任意模型都成立。

**神经网络的近似偏差可能是泛化机制的一部分，但并不是所有近似误差都有益。** [arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 九、Section 3.3：泛化究竟发生在生成过程的哪个阶段？

这是 Figure 3，也是全文最有研究价值的实验之一。

## 9.1 为什么要设计这个实验？

前面发现：

$$
v_\theta(x,t)
$$

与：

$$
\hat v^*(x,t)
$$

主要在两个时间段存在明显差异：

$$
t\approx0.15
$$

以及：

$$
t\rightarrow1
$$

那么：

**究竟哪个时间段的误差更影响最终的泛化？**

仅靠 Loss 曲线无法回答。

因此，作者设计了一个混合速度场。

---

## 9.2 构造 Hybrid Velocity Field

定义切换时间：

$$
\tau\in[0,1]
$$

混合速度场：

$$
\boxed{
v_{\mathrm{hybrid}}(x,t;\tau)
=
\begin{cases}
\hat v^*(x,t),&0\le t<\tau\\[4pt]
v_\theta(x,t),&\tau\le t\le1
\end{cases}
}
\tag{9.1}
$$

然后求解：

$$
\boxed{
\frac{dX_t}{dt}
=
v_{\mathrm{hybrid}}(X_t,t;\tau)
}
\tag{9.2}
$$

初始状态：

$$
X_0\sim\mathcal N(0,I)
$$

这相当于：

1. 从噪声开始；
2. 前面一段时间使用精确经验速度场；
3. 到达 $\tau$ 后切换成神经网络；
4. 观察最终生成结果。

---

## 9.3 分析两个极端

### 当 $\tau=0$

全程使用：

$$
v_\theta
$$

这是普通训练模型的生成过程。

它可以生成训练集之外的新图像。

### 当 $\tau=1$

全程使用：

$$
\hat v^*
$$

这是精确经验流。

理论上最终生成训练集中的某个样本。

### 当 $0<\tau<1$

前期使用精确经验速度场，后期使用神经网络。

因此，可以观察：

**如果前期轨迹已经朝某个训练样本集中，那么后期神经网络是否还能让最终结果摆脱这种记忆？**

---

## 9.4 如何衡量泛化？

作者使用 LPIPS 距离。

对于生成图像 $x$，定义：

$$
\boxed{
d(x,\mathcal D)
=
\min_{y_i\in\mathcal D}
\operatorname{LPIPS}(x,y_i)
}
\tag{9.3}
$$

其中：

$$
\operatorname{LPIPS}(x,y_i)
$$

衡量两张图像在预训练网络特征空间中的感知差异。

然后对 256 个固定初始噪声生成的图像求平均：

$$
\boxed{
G(\tau)
=
\frac1{256}
\sum_{k=1}^{256}
d(x_{\mathrm{gen}}^{(k)}(\tau),\mathcal D)
}
\tag{9.4}
$$

这里固定初始噪声非常重要。

否则不同 $\tau$ 下的结果可能受到初始随机性干扰。

---

## 9.5 Figure 3 的结果

作者在 CIFAR-10 和 CelebA-64 上发现：

当：

$$
\tau\le0.2
$$

时，混合模型仍然保持与完整神经网络模型相近的最近邻距离。

但当：

$$
\tau>0.2
$$

时，距离开始明显下降。

当：

$$
\tau\ge0.4
$$

时，生成图像通常已经非常接近全程使用精确经验速度场得到的训练样本。

因此：

$$
\boxed{
\text{生成早期的速度场近似对泛化尤其关键}
}
\tag{9.5}
$$

### 为什么这个实验比单纯比较 Loss 更有说服力？

因为它不仅观察：

$$
\|v_\theta-\hat v^*\|^2
$$

还主动改变生成过程中的速度场。

也就是说，作者不只是发现“早期误差大”，而是进一步验证：

**将早期神经网络速度场替换为精确经验速度场，会显著改变最终结果的记忆程度。**

这为早期速度场的重要性提供了更直接的证据。

不过，论文并没有证明一个普遍的时间阈值：

$$
\tau=0.2
$$

或：

$$
\tau=0.4
$$

这些是特定数据集和模型设置下的实验观察。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 十、Section 4：Empirical Flow Matching（EFM）

前面三组实验表明：

1. 高维数据中的 Target 随机性可能很快减弱；
2. 网络无法精确拟合经验速度场与泛化有关；
3. 生成早期的速度场尤其重要。

但是，作者还没有完全排除一种解释：

**虽然随机性只存在于生成早期的一小段时间，但恰好就是这段随机性帮助了模型泛化。**

所以作者需要进一步做一个直接实验：

> 在保持期望监督目标不变的情况下，降低 Target 随机性，看看生成泛化是否会下降。

这就是 EFM 的动机。

---

## 10.1 为什么不直接使用完整闭式速度场？

闭式速度为：

$$
\hat v^*(x,t)
=
\sum_{i=1}^Nw_i(x,t)\frac{y_i-x}{1-t}
$$

对于每一个训练状态 $x$，都需要计算：

$$
\|x-ty_i\|^2,\qquad i=1,\ldots,N
$$

如果：

- 训练集有 $N$ 个样本；
- 训练 batch size 为 $B$；
- 数据维度为 $D$；

那么每个 batch 的额外计算复杂度约为：

$$
\boxed{
O(BND)
}
\tag{10.1}
$$

对于 CIFAR-10：

$$
N=50000
$$

$$
D=3072
$$

每次训练都遍历整个数据集计算后验权重，成本较高。

因此，作者希望只使用：

$$
M\ll N
$$

个候选训练样本构造近似 Target。

---

# 十一、EFM Target 的完整构造过程

## 11.1 首先按照普通 CFM 采样

采样：

$$
X_0\sim\mathcal N(0,I)
$$

$$
X_1\sim\hat p_{\mathrm{data}}
$$

$$
t\sim\mathcal U(0,1)
$$

然后：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{11.1}
$$

到这里与普通 CFM 完全相同。

---

## 11.2 构造候选训练样本集合

定义：

$$
\mathcal B_M
=
\{b_1,b_2,\ldots,b_M\}
$$

其中：

$$
\boxed{
b_1=X_1
}
\tag{11.2}
$$

其余：

$$
\boxed{
b_2,\ldots,b_M
\overset{\mathrm{i.i.d.}}{\sim}
\hat p_{\mathrm{data}}
}
\tag{11.3}
$$

**这里 $b_1=X_1$ 是整个 EFM 算法最重要的设计。**

它不能随意替换成另一个独立采样的训练样本。

原因稍后通过 Proposition 2 的无偏性证明解释。

---

## 11.3 在候选集合内计算后验权重

对于每个候选样本：

$$
b_j
$$

计算：

$$
a_j(x,t)
=
-\frac{\|x-tb_j\|^2}{2(1-t)^2}
$$

然后：

$$
\boxed{
\tilde w_j(x,t)
=
\frac{
\exp(a_j(x,t))
}{
\sum_{k=1}^M\exp(a_k(x,t))
}
}
\tag{11.4}
$$

满足：

$$
\sum_{j=1}^M\tilde w_j(x,t)=1
$$

注意：

$$
\tilde w_j
$$

只是在当前 $M$ 个候选样本之间归一化。

它不是完整训练集上的后验概率：

$$
w_i
$$

---

## 11.4 构造 EFM Target

对于每个候选样本：

$$
b_j
$$

对应条件速度：

$$
v_j(x,t)
=
\frac{b_j-x}{1-t}
$$

使用局部 Softmax 权重求平均：

$$
\boxed{
\hat v_M(x,t)
=
\sum_{j=1}^M
\tilde w_j(x,t)
\frac{b_j-x}{1-t}
}
\tag{11.5}
$$

这就是 EFM Target。

训练损失：

$$
\boxed{
\mathcal L_{\mathrm{EFM}}(\theta)
=
\mathbb E
\left[
\|v_\theta(X_t,t)-\hat v_M(X_t,t)\|^2
\right]
}
\tag{11.6}
$$

其中 $\hat v_M$ 不需要反向传播到候选样本。

模型仍然只进行一次正常的网络预测：

$$
v_\theta(X_t,t)
$$

因此，EFM 的额外开销主要来自 Target 构造，而不是增加网络前向次数。

---

## 11.5 为什么 $M=1$ 时退化为普通 CFM？

当：

$$
M=1
$$

候选集合只有：

$$
b_1=X_1
$$

Softmax 权重：

$$
\tilde w_1=1
$$

所以：

$$
\begin{aligned}
\hat v_1(X_t,t)
&=
\frac{X_1-X_t}{1-t}\\
&=
\frac{
X_1-(1-t)X_0-tX_1
}{1-t}\\
&=X_1-X_0
\end{aligned}
$$

因此：

$$
\boxed{
\hat v_1(X_t,t)=V
}
\tag{11.7}
$$

即：

$$
\boxed{
\mathrm{EFM}(M=1)=\mathrm{CFM}
}
\tag{11.8}
$$

所以 EFM 并不是完全不同的生成模型，而是对 CFM Target 的一种推广。

通过改变 $M$，可以改变 Target 的随机性。

---

# 十二、Proposition 2：为什么 EFM Target 是无偏的？

这是论文最值得认真理解的数学证明。

作者证明：

$$
\boxed{
\mathbb E[\hat v_M(x,t)\mid X_t=x]
=
\hat v^*(x,t)
}
\tag{12.1}
$$

也就是说，虽然每次只使用 $M$ 个候选样本，但在条件期望意义下，EFM Target 仍然等于完整经验速度场。

## 12.1 首先固定当前状态

固定：

$$
X_t=x
$$

以及时间：

$$
t
$$

定义完整训练集上的后验权重：

$$
\boxed{
q_i=w_i(x,t)
}
\tag{12.2}
$$

因此：

$$
\sum_{i=1}^Nq_i=1
$$

再定义每个训练样本对应的条件速度：

$$
\boxed{
a_i=\frac{y_i-x}{1-t}
}
\tag{12.3}
$$

那么精确经验速度场：

$$
\boxed{
\hat v^*(x,t)
=
\sum_{i=1}^Nq_i a_i
}
\tag{12.4}
$$

现在我们希望证明 EFM 的期望也等于它。

---

## 12.2 用样本索引表示 EFM

设：

$$
I_1,\ldots,I_M
$$

表示候选训练样本的索引。

因为：

$$
b_1=X_1
$$

所以给定：

$$
X_t=x
$$

后：

$$
\boxed{
I_1\sim\operatorname{Cat}(q_1,\ldots,q_N)
}
\tag{12.5}
$$

而其余候选样本独立采样于经验分布：

$$
\boxed{
I_2,\ldots,I_M
\overset{\mathrm{i.i.d.}}{\sim}
\operatorname{Unif}\{1,\ldots,N\}
}
\tag{12.6}
$$

因此，EFM Target 可以写成：

$$
\boxed{
\hat v_M
=
\frac{
\sum_{j=1}^Mq_{I_j}a_{I_j}
}{
\sum_{j=1}^Mq_{I_j}
}
}
\tag{12.7}
$$

为什么这里可以使用 $q_{I_j}$？

因为：

$$
q_i
=
\frac{e^{a_i^{\mathrm{score}}}}
{\sum_{k=1}^Ne^{a_k^{\mathrm{score}}}}
$$

其中：

$$
a_i^{\mathrm{score}}
=
-\frac{\|x-ty_i\|^2}{2(1-t)^2}
$$

EFM 只在候选集合内部归一化，所以完整训练集 Softmax 的公共分母会再次抵消。

因此，式 (12.7) 与式 (11.5) 完全等价。

---

## 12.3 写出期望

因为：

$$
I_1\sim\operatorname{Cat}(q)
$$

其余索引均匀采样，所以：

$$
\boxed{
\mathbb E[\hat v_M]
=
\sum_{i_1,\ldots,i_M}
\frac{q_{i_1}}{N^{M-1}}
\frac{
\sum_{j=1}^Mq_{i_j}a_{i_j}
}{
\sum_{j=1}^Mq_{i_j}
}
}
\tag{12.8}
$$

这个公式看起来复杂，但有一个关键性质：

$$
\frac{
\sum_jq_{i_j}a_{i_j}
}{
\sum_jq_{i_j}
}
$$

对候选样本的位置排列是对称的。

也就是说，交换：

$$
i_1,i_2,\ldots,i_M
$$

的顺序，不会改变加权平均结果。

因此，在对所有索引组合求和时，第一个位置可以与其他位置交换。

于是：

$$
\begin{aligned}
\mathbb E[\hat v_M]
&=
\frac1{MN^{M-1}}
\sum_{i_1,\ldots,i_M}
\left(
\sum_{k=1}^Mq_{i_k}
\right)
\frac{
\sum_{j=1}^Mq_{i_j}a_{i_j}
}{
\sum_{j=1}^Mq_{i_j}
}
\end{aligned}
\tag{12.9}
$$

现在可以直接约去：

$$
\sum_{k=1}^Mq_{i_k}
$$

得到：

$$
\boxed{
\mathbb E[\hat v_M]
=
\frac1{MN^{M-1}}
\sum_{i_1,\ldots,i_M}
\sum_{j=1}^Mq_{i_j}a_{i_j}
}
\tag{12.10}
$$

交换求和顺序：

$$
\begin{aligned}
\mathbb E[\hat v_M]
&=
\frac1{MN^{M-1}}
\sum_{j=1}^M
\sum_{i_1,\ldots,i_M}
q_{i_j}a_{i_j}
\end{aligned}
$$

对于固定的 $j$，其余 $M-1$ 个索引共有：

$$
N^{M-1}
$$

种组合。

因此：

$$
\begin{aligned}
\mathbb E[\hat v_M]
&=
\frac1{MN^{M-1}}
\sum_{j=1}^M
N^{M-1}
\sum_{i=1}^Nq_i a_i\\
&=
\frac1M
\sum_{j=1}^M
\sum_{i=1}^Nq_i a_i\\
&=
\sum_{i=1}^Nq_i a_i
\end{aligned}
$$

最终：

$$
\boxed{
\mathbb E[\hat v_M]
=
\hat v^*(x,t)
}
\tag{12.11}
$$

无偏性得证。

### 这里最关键的地方是什么？

不是 Softmax 本身。

而是候选样本的特殊采样方式：

$$
\boxed{
I_1\sim\operatorname{Cat}(q)
}
$$

$$
\boxed{
I_2,\ldots,I_M\sim\operatorname{Unif}
}
$$

如果全部候选样本都从均匀经验分布独立采样，那么这种自归一化加权平均一般是有偏的。

**EFM 的无偏性依赖于始终包含生成当前 $X_t$ 的真实终点 $X_1$。**

这也是作者在正文和附录中反复强调的细节。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 十三、Proposition 2：为什么 EFM 的 Target 方差不增？

作者还证明：

$$
\boxed{
\operatorname{Var}(\hat v_M\mid X_t=x)
\le
\operatorname{Var}(V\mid X_t=x)
}
\tag{13.1}
$$

下面给出向量形式的证明。

## 13.1 定义两个随机变量

普通 CFM Target：

$$
V=a_{I_1}
$$

其中：

$$
I_1\sim\operatorname{Cat}(q)
$$

EFM Target：

$$
\hat v_M
=
\frac{
\sum_jq_{I_j}a_{I_j}
}{
\sum_jq_{I_j}
}
$$

根据前面的证明：

$$
\mathbb E[V]
=
\mathbb E[\hat v_M]
=
\hat v^*
$$

所以，只需要比较二阶矩：

$$
\mathbb E[\|\hat v_M\|^2]
$$

和：

$$
\mathbb E[\|V\|^2]
$$

即可。

---

## 13.2 利用平方范数的凸性

对于任意非负权重：

$$
\alpha_j\ge0
$$

且：

$$
\sum_j\alpha_j=1
$$

由 Jensen 不等式：

$$
\boxed{
\left\|
\sum_j\alpha_j a_j
\right\|^2
\le
\sum_j\alpha_j\|a_j\|^2
}
\tag{13.2}
$$

对于 EFM：

$$
\alpha_j
=
\frac{q_{I_j}}{\sum_kq_{I_k}}
$$

因此：

$$
\boxed{
\|\hat v_M\|^2
\le
\frac{
\sum_jq_{I_j}\|a_{I_j}\|^2
}{
\sum_jq_{I_j}
}
}
\tag{13.3}
$$

右边仍然是一个自归一化加权平均。

根据前面完全相同的无偏性推导，只需要将：

$$
a_i
$$

替换为：

$$
\|a_i\|^2
$$

就得到：

$$
\mathbb E
\left[
\frac{
\sum_jq_{I_j}\|a_{I_j}\|^2
}{
\sum_jq_{I_j}
}
\right]
=
\sum_iq_i\|a_i\|^2
$$

因此：

$$
\boxed{
\mathbb E[\|\hat v_M\|^2]
\le
\sum_iq_i\|a_i\|^2
}
\tag{13.4}
$$

而普通 CFM：

$$
\mathbb E[\|V\|^2]
=
\sum_iq_i\|a_i\|^2
$$

所以：

$$
\boxed{
\mathbb E[\|\hat v_M\|^2]
\le
\mathbb E[\|V\|^2]
}
\tag{13.5}
$$

---

## 13.3 转化为方差不等式

向量总方差：

$$
\operatorname{Var}(Z)
=
\mathbb E[\|Z\|^2]
-
\|\mathbb E[Z]\|^2
$$

由于：

$$
\mathbb E[\hat v_M]
=
\mathbb E[V]
$$

因此从式 (13.5) 得到：

$$
\boxed{
\operatorname{Var}(\hat v_M)
\le
\operatorname{Var}(V)
}
\tag{13.6}
$$

证明完成。

这里的方差比较都是在固定 $(x,t)$ 的条件下进行的。

### 一个重要的理解

EFM 并不是简单地把 $M$ 个随机条件速度求算术平均：

$$
\frac1M\sum_{j=1}^MV_j
$$

而是：

$$
\boxed{
\hat v_M
=
\sum_j\tilde w_jV_j
}
$$

其中权重来自当前状态的高斯后验似然。

因此，EFM 利用了当前状态 $x$ 与候选训练样本之间的统计关系。

这也是它与普通增加 batch size 的区别。

---

# 十四、EFM 的期望 Loss 与 CFM 有什么关系？

根据前面的正交分解：

$$
\mathcal L_{\mathrm{EFM}}(\theta)
=
\mathbb E
\left[
\|v_\theta(X_t,t)-\hat v_M(X_t,t)\|^2
\right]
$$

因为：

$$
\mathbb E[\hat v_M\mid X_t,t]
=
\hat v^*(X_t,t)
$$

所以：

$$
\boxed{
\mathcal L_{\mathrm{EFM}}(\theta)
=
\mathcal L_{\mathrm{FM}}(\theta)
+
C_M
}
\tag{14.1}
$$

其中：

$$
\boxed{
C_M
=
\mathbb E
\left[
\|\hat v_M-\hat v^*\|^2
\right]
}
\tag{14.2}
$$

而普通 CFM：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}(\theta)
=
\mathcal L_{\mathrm{FM}}(\theta)
+
C_1
}
\tag{14.3}
$$

根据方差不增：

$$
\boxed{
C_M\le C_1
}
\tag{14.4}
$$

因此：

$$
\boxed{
\mathcal L_{\mathrm{EFM}}(\theta)
\le
\mathcal L_{\mathrm{CFM}}(\theta)
}
\tag{14.5}
$$

这里比较的是**期望平方损失**，而且两种 Loss 使用同一个模型 $v_\theta$。

需要注意：

$$
\mathcal L_{\mathrm{EFM}}
\le
\mathcal L_{\mathrm{CFM}}
$$

并不意味着 EFM 的神经网络一定训练得更好。

因为这个不等式中包含了与模型无关的 Target 方差常数。

真正重要的是：

$$
\boxed{
\arg\min_\theta
\mathcal L_{\mathrm{EFM}}
=
\arg\min_\theta
\mathcal L_{\mathrm{CFM}}
}
\tag{14.6}
$$

同时，EFM 的监督目标具有更低或相同的条件方差。

不过，也不能由此直接推出 EFM 的**参数梯度方差**在任意网络和优化器下都严格更小；Proposition 2 直接保证的是 Target 的条件方差。

---

# 十五、Section 4.2：EFM 的实际实验结果

作者使用：

- CIFAR-10；
- CelebA-64。

比较：

$$
\mathrm{CFM}
$$

$$
\mathrm{OTCFM}
$$

以及：

$$
\mathrm{EFM}(M)
$$

其中：

$$
\boxed{
M\in\{128,256,1000\}
}
\tag{15.1}
$$

这里 OTCFM 是采用 minibatch optimal transport coupling 的 Flow Matching 基线。

需要注意，OTCFM 与 EFM 的机制不同：

- OTCFM 改变样本耦合方式；
- EFM 改变速度 Target 的估计方式。

不能把它们看成同一种降方差方法。

---

## 15.1 评价指标

作者使用两种特征空间：

**Inception-V3 特征：**

传统 FID。

**DINOv2 特征：**

用于补充传统 Inception FID 的不足。

同时分别计算：

$$
\mathrm{FID}_{\mathrm{train}}
$$

和：

$$
\mathrm{FID}_{\mathrm{test}}
$$

即生成分布分别与训练集、测试集之间的距离。

这样可以观察：

**降低 Target 随机性之后，模型是否只是更好地拟合训练集，还是对测试集的统计匹配也有所改善。**

---

## 15.2 Figure 4 的核心结果

作者发现：

随着：

$$
M=128\rightarrow256\rightarrow1000
$$

EFM 的生成质量没有下降。

相反，在多个评价设置下，性能表现出小幅但持续的改善。

这说明：

$$
\boxed{
\text{降低 Target 随机性}
\not\Rightarrow
\text{泛化能力下降}
}
\tag{15.2}
$$

这是对论文标题最直接的实验证据。

但要注意：

Proposition 2 只保证 EFM Target 的方差不超过普通 CFM，并不保证方差随 $M$ 对每个样本严格单调下降。

Figure 4 中随 $M$ 增大而改善，是实验观察。

---

## 15.3 EFM 的计算成本

附录 D.1 给出了计算时间分析。

作者在 NVIDIA L4 GPU 上测量，发现 EFM 额外计算成本相对较低。

例如 batch size 为 256 时：

- EFM-128 的额外开销不到 30 ms；
- EFM-1000 的额外开销约为 230 ms。

原因是 EFM 不需要为每个候选训练样本运行一次 U-Net。

它只需要计算：

$$
\|x-tb_j\|^2
$$

和对应的加权平均。

模型的前向和反向传播次数保持不变。

因此，EFM 的主要额外成本是：

$$
\boxed{
O(BMD)
}
$$

而不是 $M$ 倍神经网络训练成本。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 十六、附录中的补充实验与实现细节

这部分虽然不是论文主线，但对复现实验有帮助。

## 16.1 MNIST 实验

附录 C 还使用 MNIST 进行实验。

作者发现，与 CIFAR-10 和 CelebA 类似，使用更确定的速度 Target 并没有损害生成泛化。

部分情况下，模型更早达到较好的测试集 FID。

但 MNIST 的数据结构相对简单，不能仅凭该实验推断复杂高维生成任务的所有性质。

---

## 16.2 训练超参数

附录 D 给出的主要训练参数：

| 参数 | 设置 |
|---|---|
| U-Net 基础通道数 | 128 |
| Batch Size | 128 |
| Learning Rate | $2\times10^{-4}$ |
| EMA Decay | 0.9999 |
| Gradient Clipping | 1 |

Figure 2 使用约 3400 万参数的 U-Net。

Figure 2 的训练集大小消融使用 50,000 次训练迭代。

Figure 4 的方法比较使用更长的训练过程，正文附录报告为 500,000 次迭代。

因此，Figure 2 与 Figure 4 不能直接比较绝对生成性能，因为实验目的和训练预算不同。

---

# 十七、全文理论与实验的逻辑闭环

现在可以将论文重新组织为一个完整的论证过程。

## 第一步：找到精确经验速度场

通过贝叶斯公式：

$$
\boxed{
\hat v^*(x,t)
=
\sum_iw_i(x,t)\frac{y_i-x}{1-t}
}
$$

这是 Proposition 1。

## 第二步：发现精确经验速度场存在记忆问题

因为：

$$
p_1=\hat p_{\mathrm{data}}
$$

所以精确经验流最终会复现训练集。

于是提出问题：

**为什么实际神经网络能够泛化？**

## 第三步：检查 Target 随机性

比较：

$$
V
$$

与：

$$
\hat v^*(X_t,t)
$$

发现高维图像中两者很早就高度对齐。

因此，Target 随机性可能没有想象中那么重要。

## 第四步：检查网络与精确经验速度场的差异

固定网络容量，增加训练集大小。

发现：

$$
\|v_\theta-\hat v^*\|^2
$$

在特定时间段增大。

同时，模型逐渐表现出更好的泛化能力。

## 第五步：定位泛化相关的时间段

使用 Hybrid Flow：

$$
\hat v^*
\rightarrow
v_\theta
$$

通过改变切换时间：

$$
\tau
$$

发现生成早期的速度场对最终是否记忆训练集具有重要影响。

## 第六步：直接降低 Target 随机性

提出 EFM：

$$
\hat v_M
$$

证明：

$$
\mathbb E[\hat v_M\mid X_t,t]=\hat v^*
$$

以及：

$$
\operatorname{Var}(\hat v_M\mid X_t,t)
\le
\operatorname{Var}(V\mid X_t,t)
$$

## 第七步：验证泛化是否下降

在 CIFAR-10、CelebA 和补充的 MNIST 实验中，发现降低 Target 随机性没有损害生成质量。

于是支持论文标题中的核心观点：

$$
\boxed{
\text{Generalization Does Not Arise from Target Stochasticity}
}
$$

更准确地说，是：

**在作者研究的图像生成设置中，Target 随机性不是解释 Flow Matching 泛化现象所必需的关键因素。**

---

# 十八、这篇论文有哪些尚未解决的问题？

我认为有四个需要特别注意的地方。

### 1. 没有真正解释神经网络为什么泛化

作者证明：

$$
v_\theta\neq\hat v^*
$$

与泛化相关。

但没有进一步完整刻画：

$$
v_\theta
$$

究竟学到了什么。

例如，网络是否学习了：

- 数据的局部结构；
- 图像块之间的统计规律；
- 某种平滑化后的速度场；
- 低维流形附近的输运结构。

这些仍然需要进一步研究。

论文 Related Work 中提到了从图像 Patch 结构解释生成泛化的相关研究，但没有将其发展为本文的完整理论。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

### 2. 没有证明所有 Target 随机性都没有作用

EFM 实验主要针对标准线性 CFM 的条件速度随机性。

它没有证明：

- 所有噪声分布；
- 所有概率路径；
- 所有网络架构；
- 所有数据维度；

都不需要 Target 随机性。

因此，标题应结合实验范围理解。

### 3. 没有证明“近似误差越大越好”

如果：

$$
v_\theta
$$

与：

$$
\hat v^*
$$

相差太大，模型同样可能无法生成合理数据。

泛化需要的是有结构的近似，而不是任意误差。

### 4. EFM 不是直接解决泛化理论的最终答案

EFM 的贡献是：

**提供一种保持期望监督目标不变、同时降低 Target 方差的训练方式。**

它能够帮助检验 Target 随机性的作用。

但它没有从数学上直接推导出生成模型的泛化误差界。

作者也明确承认，网络架构的归纳偏置与优化过程之间的相互作用仍然没有得到完整解释。[arXiv](https://arxiv.org/html/2506.03719?utm_source=chatgpt.com)

---

# 十九、与 MeanFlow、Shortcut 的关系

结合我们之前讨论的内容，这篇论文有一个很有价值的启发。

普通 FM 学习：

$$
\boxed{
v^*(x,t)
=
\mathbb E[X_1-X_0\mid X_t=x]
}
$$

MeanFlow 和 Shortcut 则希望学习区间平均速度：

$$
\boxed{
u^*(x,t,d)
=
\frac1d
\int_t^{t+d}
v^*(x_\tau,\tau)\,d\tau
}
\tag{19.1}
$$

这里的 $x_\tau$ 是沿着边缘 ODE 演化的状态。

因此，三者在理论上研究的是同一概率流的不同对象：

| 方法 | 学习对象 | 主要问题 |
|---|---|---|
| FM | 瞬时边缘速度 | 如何学习概率流 |
| MeanFlow | 区间平均速度 | 如何一步生成 |
| Shortcut | 区间平均速度 | 如何通过离散自一致性学习多步与一步映射 |
| Closed-Form FM / EFM | 瞬时边缘速度及其估计 | 为什么能够泛化、如何降低 Target 方差 |

这带来一个很有意思的研究问题：

**如果使用 EFM 的低方差速度监督作为 MeanFlow 或 Shortcut 的基础监督，是否能够提高平均速度学习的稳定性？**

例如，MeanFlow 中需要用到瞬时速度 Target：

$$
V=X_1-X_0
$$

那么可以研究将其替换为：

$$
\hat v_M(X_t,t)
$$

会产生什么影响。

但这里必须强调：

**这只是一个值得研究的方向，并不是直接替换就一定正确。**

因为 MeanFlow 的训练 Target 还涉及速度场的方向导数、JVP 和停止梯度操作。

将随机条件速度替换为其条件期望，可能改变半梯度训练的性质，需要重新推导。

因此，不能直接利用 EFM 的无偏性证明 MeanFlow 的新训练目标也完全等价。

---

# 二十、最终总结

我认为，这篇论文最值得记住的不是 EFM 本身，而是它对生成模型泛化问题的重新理解。

过去一种直觉是：

$$
\text{随机 Target}
\rightarrow
\text{随机梯度}
\rightarrow
\text{正则化}
\rightarrow
\text{泛化}
$$

但这篇论文发现，在高维图像生成任务中，这条解释链并不充分。

作者给出了另一种更符合实验现象的解释：

$$
\boxed{
\begin{gathered}
\text{有限训练集定义复杂的经验速度场}\\
\Downarrow\\
\text{精确经验速度场会记忆训练样本}\\
\Downarrow\\
\text{有限容量神经网络无法完全拟合它}\\
\Downarrow\\
\text{网络形成具有归纳偏置的近似速度场}\\
\Downarrow\\
\text{生成结果可能表现出泛化能力}
\end{gathered}
}
$$

其中最后两步仍然是有待进一步理论刻画的解释，而不是论文已经证明的普遍定理。

**全文最重要的三个数学结果：**

$$
\boxed{
\hat v^*(x,t)
=
\sum_iw_i(x,t)\frac{y_i-x}{1-t}
}
$$

$$
\boxed{
\mathbb E[\hat v_M\mid X_t,t]
=
\hat v^*(X_t,t)
}
$$

$$
\boxed{
\operatorname{Var}(\hat v_M\mid X_t,t)
\le
\operatorname{Var}(V\mid X_t,t)
}
$$

**全文最重要的三个实验发现：**

1. 高维数据中，条件速度与边缘速度在生成早期就迅速对齐。
2. 神经网络对精确经验速度场的不完全拟合与泛化现象相关，早期速度场尤其重要。
3. 使用 EFM 降低 Target 随机性，并没有损害生成性能。

最终，这篇论文并没有完整回答“生成模型为什么能够泛化”，但它通过闭式速度场和有针对性的实验，**排除了一个此前受到关注的解释，并将问题进一步指向网络架构、有限容量和优化过程产生的归纳偏置。**

这也是它最主要的理论价值。